# Notebook 14 — Phase Alignment Recovery

This notebook extends Notebook 13 from **recovery time** to **recovery quality**.

Notebook 13 asks:

> how long does it take to return above the 45° CGCS threshold?

Notebook 14 asks:

> after returning above threshold, does the policy recover cleanly, or does it re-enter with overshoot, ringing, or unstable alignment?

Outputs are written to:

```text
figures/phase_alignment_recovery/
results/phase_alignment_recovery_summary.json
results/phase_alignment_recovery_summary.md
docs/phase_alignment_recovery.md
```


In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(9423)

NOTEBOOK_ID = "14"
SLUG = "phase_alignment_recovery"
PREFIX = f"{NOTEBOOK_ID}_{SLUG}"

FIG_DIR = f"figures/{SLUG}"
RESULTS_DIR = "results"
DOCS_DIR = "docs"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

def save_fig(name):
    path_png = f"{FIG_DIR}/{PREFIX}_{name}.png"
    path_pdf = f"{FIG_DIR}/{PREFIX}_{name}.pdf"
    plt.savefig(path_png, dpi=220, bbox_inches="tight")
    plt.savefig(path_pdf, bbox_inches="tight")
    print(f"[saved] {path_png}")
    print(f"[saved] {path_pdf}")

threshold = 1 / math.sqrt(1**2 + 1**2)
threshold


## 1. Shared disturbance environment

Use a deliberately harsh environment similar to Notebooks 12–13: drift, noise burst, model mismatch, and shock events.


In [ ]:
# -----------------------------
# Simulation configuration
# -----------------------------
N = 80
blocks = np.arange(N)
pulse_t = np.linspace(0, 10, 320)

noise_burst = (28, 40)
model_mismatch = (52, 66)
shock_blocks = [26, 39, 52, 69, 75]

# True calibration drifts.
omega_true = (
    0.26 * np.sin(0.35 * blocks)
    + 0.13 * np.sin(0.92 * blocks + 0.6)
    + 0.0045 * blocks
)

b_true = (
    0.0025 * blocks
    + 0.028 * np.sin(0.21 * blocks + 0.3)
    + 0.018 * np.sin(0.63 * blocks)
)

# Model mismatch: true system shifts, measurements partially obscure it.
omega_true[model_mismatch[0]:model_mismatch[1]] += np.linspace(0.0, 0.55, model_mismatch[1] - model_mismatch[0])
b_true[model_mismatch[0]:model_mismatch[1]] += np.linspace(0.0, 0.28, model_mismatch[1] - model_mismatch[0])

# Corrupted measurements.
omega_meas = omega_true + np.random.normal(0, 0.018, N)
b_meas = b_true + np.random.normal(0, 0.012, N)

# Noise burst region.
omega_meas[noise_burst[0]:noise_burst[1]] += np.random.normal(0, 0.09, noise_burst[1] - noise_burst[0])
b_meas[noise_burst[0]:noise_burst[1]] += np.random.normal(0, 0.055, noise_burst[1] - noise_burst[0])

# Shock events.
for s in shock_blocks:
    omega_meas[s] += np.random.choice([-1, 1]) * np.random.uniform(0.35, 0.85)
    b_meas[s] += np.random.choice([-1, 1]) * np.random.uniform(0.16, 0.42)

# Response model:
# target is a smooth excited-state probability curve.
def response_curve(t, omega, b):
    y = np.sin((1.0 + omega) * t / 2.0) ** 2 + b
    return np.clip(y, 0, 1.05)

target_response = response_curve(pulse_t, 0.0, 0.0)

def response_rmse(omega_cmd, b_cmd):
    rmses = []
    for o, b in zip(omega_cmd, b_cmd):
        y = response_curve(pulse_t, o, b)
        rmses.append(float(np.sqrt(np.mean((y - target_response) ** 2))))
    return np.array(rmses)

def cosine_similarity_to_target(omega_cmd, b_cmd):
    # Compare response vector to target response vector.
    sims = []
    target_norm = np.linalg.norm(target_response)
    for o, b in zip(omega_cmd, b_cmd):
        y = response_curve(pulse_t, o, b)
        denom = np.linalg.norm(y) * target_norm
        sims.append(float(np.dot(y, target_response) / denom) if denom > 0 else 0.0)
    return np.array(sims)

def moving_average(x, w=5):
    out = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - w + 1)
        out[i] = np.mean(x[lo:i+1])
    return out


## 2. Policy definitions

The policies are intentionally simple and transparent. The point is not hardware optimality; the point is comparing re-entry quality after threshold failures.


In [ ]:
# -----------------------------
# Simple filter implementations
# -----------------------------
def scalar_kalman(z, q=5e-4, r=2e-3, x0=0.0, p0=1.0):
    x = x0
    p = p0
    xs = []
    for zi in z:
        # predict
        p = p + q
        # update
        k = p / (p + r)
        x = x + k * (zi - x)
        p = (1 - k) * p
        xs.append(x)
    return np.array(xs)

def robust_gated_kalman(z, q=6e-4, r=2e-3, gate=0.42, x0=0.0, p0=1.0):
    x = x0
    p = p0
    xs = []
    for zi in z:
        p = p + q
        residual = zi - x
        if abs(residual) > gate:
            # Down-weight obvious shock instead of fully trusting it.
            r_eff = r * 40.0
        else:
            r_eff = r
        k = p / (p + r_eff)
        x = x + k * residual
        p = (1 - k) * p
        xs.append(x)
    return np.array(xs)

def joint_kalman(z_omega, z_b, q_omega=6e-4, q_b=2e-4, q_cross=2e-5, r_omega=2e-3, r_b=1.2e-3):
    x = np.array([0.0, 0.0])
    P = np.eye(2)
    Q = np.array([[q_omega, q_cross], [q_cross, q_b]])
    R = np.array([[r_omega, 0.0], [0.0, r_b]])
    H = np.eye(2)
    xs = []
    for zo, zb in zip(z_omega, z_b):
        z = np.array([zo, zb])
        P = P + Q
        S = H @ P @ H.T + R
        K = P @ H.T @ np.linalg.inv(S)
        x = x + K @ (z - H @ x)
        P = (np.eye(2) - K @ H) @ P
        xs.append(x.copy())
    return np.array(xs)

def constrained_mpc_lite(omega_est, b_est, max_step_o=0.18, max_step_b=0.075, smoothing=0.55):
    # Smooth step-limited command; tracks estimate but avoids violent jumps.
    o_cmd = np.zeros_like(omega_est)
    b_cmd = np.zeros_like(b_est)
    for i in range(len(omega_est)):
        if i == 0:
            desired_o, desired_b = omega_est[i], b_est[i]
            o_cmd[i] = np.clip(desired_o, -max_step_o, max_step_o)
            b_cmd[i] = np.clip(desired_b, -max_step_b, max_step_b)
        else:
            desired_o = smoothing * omega_est[i] + (1 - smoothing) * o_cmd[i-1]
            desired_b = smoothing * b_est[i] + (1 - smoothing) * b_cmd[i-1]
            o_cmd[i] = o_cmd[i-1] + np.clip(desired_o - o_cmd[i-1], -max_step_o, max_step_o)
            b_cmd[i] = b_cmd[i-1] + np.clip(desired_b - b_cmd[i-1], -max_step_b, max_step_b)
    return o_cmd, b_cmd

# Estimates / commands.
ma_o = moving_average(omega_meas, w=6)
ma_b = moving_average(b_meas, w=6)

scalar_o = scalar_kalman(omega_meas, q=7e-4, r=2e-3)
scalar_b = scalar_kalman(b_meas, q=2.5e-4, r=1.3e-3)

robust_o = robust_gated_kalman(omega_meas, q=7e-4, r=2e-3, gate=0.42)
robust_b = robust_gated_kalman(b_meas, q=2.5e-4, r=1.3e-3, gate=0.22)

joint_est = joint_kalman(omega_meas, b_meas)
joint_o, joint_b = joint_est[:, 0], joint_est[:, 1]

constrained_o, constrained_b = constrained_mpc_lite(joint_o, joint_b)

# Policy command dictionary: command is negative compensation.
policies = {
    "none": (np.zeros(N), np.zeros(N)),
    "moving_average": (-ma_o, -ma_b),
    "scalar_kalman": (-scalar_o, -scalar_b),
    "joint_kalman": (-joint_o, -joint_b),
    "robust_gated_kalman": (-robust_o, -robust_b),
    "constrained_mpc": (-constrained_o, -constrained_b),
    "oracle": (-omega_true, -b_true),
}

# Residual drift after command.
residuals = {}
for name, (co, cb) in policies.items():
    residuals[name] = {
        "omega": omega_true + co,
        "b": b_true + cb,
    }

rmse = {name: response_rmse(v["omega"], v["b"]) for name, v in residuals.items()}
cosines = {name: cosine_similarity_to_target(v["omega"], v["b"]) for name, v in residuals.items()}
margins = {name: cosines[name] - threshold for name in policies}


## 3. Failure → recovery event detection

A recovery event starts when a policy moves from below threshold back above threshold. Notebook 14 measures the local quality after that re-entry.


In [ ]:
def failure_events(cosine, thr=threshold):
    below = cosine < thr
    events = []
    i = 0
    n = len(cosine)
    while i < n:
        if below[i]:
            start = i
            while i < n and below[i]:
                i += 1
            end = i - 1
            recovered_at = i if i < n else None
            events.append({"start": start, "end": end, "duration": end - start + 1, "recovered_at": recovered_at})
        else:
            i += 1
    return events

def recovery_quality_metrics(cosine, rmse_series, thr=threshold, horizon=8):
    events = failure_events(cosine, thr)
    recovered = [e for e in events if e["recovered_at"] is not None]
    slopes = []
    overshoots = []
    integrals = []
    ringing = []
    settling_blocks = []

    for e in recovered:
        r = e["recovered_at"]
        end = min(len(cosine), r + horizon)
        post = cosine[r:end]
        post_rmse = rmse_series[r:end]
        if len(post) == 0:
            continue

        # Slope immediately after re-entry.
        if len(post) >= 2:
            slopes.append(float(post[1] - post[0]))
        else:
            slopes.append(0.0)

        # Overshoot means drop back toward threshold after first recovery point.
        local_margin = post - thr
        overshoots.append(float(max(0.0, local_margin[0] - np.min(local_margin))))

        # Integral of phase error after re-entry.
        integrals.append(float(np.sum(np.maximum(0.0, 1.0 - post))))

        # Ringing: mean absolute second difference of cosine after recovery.
        if len(post) >= 3:
            ringing.append(float(np.mean(np.abs(np.diff(post, n=2)))))
        else:
            ringing.append(0.0)

        # Settling: blocks until cosine remains above 0.98 for rest of local horizon.
        settle = len(post)
        for k in range(len(post)):
            if np.all(post[k:] >= 0.98):
                settle = k
                break
        settling_blocks.append(float(settle))

    return {
        "failure_count": int(len(events)),
        "recovery_count": int(len(recovered)),
        "time_below_threshold": int(np.sum(cosine < thr)),
        "post_recovery_cosine_slope": float(np.mean(slopes)) if slopes else 0.0,
        "phase_alignment_error_integral": float(np.mean(integrals)) if integrals else 0.0,
        "reentry_overshoot": float(np.mean(overshoots)) if overshoots else 0.0,
        "ringing_score": float(np.mean(ringing)) if ringing else 0.0,
        "settling_blocks": float(np.mean(settling_blocks)) if settling_blocks else 0.0,
        "events": events,
    }

metrics = {}
for name in policies:
    if name == "oracle":
        # Keep oracle in final ranking as ideal reference.
        metrics[name] = {
            "failure_count": 0,
            "recovery_count": 0,
            "time_below_threshold": 0,
            "post_recovery_cosine_slope": 0.0,
            "phase_alignment_error_integral": 0.0,
            "reentry_overshoot": 0.0,
            "ringing_score": 0.0,
            "settling_blocks": 0.0,
            "events": [],
        }
    else:
        metrics[name] = recovery_quality_metrics(cosines[name], rmse[name], horizon=8)

# Clean recovery score: higher is better.
# Penalize failures, time below threshold, post-recovery error, overshoot, ringing, and settling delay.
for name, m in metrics.items():
    mean_margin = float(np.mean(margins[name]))
    clean_score = (
        mean_margin
        - 0.10 * m["failure_count"]
        - 0.020 * m["time_below_threshold"]
        - 0.50 * m["phase_alignment_error_integral"]
        - 0.70 * m["reentry_overshoot"]
        - 0.30 * m["ringing_score"]
        - 0.030 * m["settling_blocks"]
    )
    m["mean_threshold_margin"] = mean_margin
    m["clean_recovery_score"] = float(clean_score)

ranking = sorted(metrics.keys(), key=lambda k: metrics[k]["clean_recovery_score"], reverse=True)
ranking


## 4. Figures


In [ ]:
plt.figure(figsize=(14, 6))
for name in ["none", "moving_average", "scalar_kalman", "joint_kalman", "robust_gated_kalman", "constrained_mpc", "oracle"]:
    plt.plot(blocks, cosines[name], marker="o", label=name)
plt.axhline(threshold, linestyle="--", label="45° threshold")
plt.axvspan(noise_burst[0], noise_burst[1], alpha=0.15, label="noise burst")
plt.axvspan(model_mismatch[0], model_mismatch[1], alpha=0.10, label="model mismatch")
plt.title("Phase alignment recovery: CGCS phase-lock stability")
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.ylim(0.55, 1.01)
plt.legend(loc="lower left")
save_fig("cgcs_stability_comparison")
plt.show()


In [ ]:
plt.figure(figsize=(14, 6))
for name in ["none", "moving_average", "scalar_kalman", "joint_kalman", "robust_gated_kalman", "constrained_mpc"]:
    plt.plot(blocks, margins[name], marker="o", label=name)
plt.axhline(0, linestyle="--", label="threshold margin = 0")
plt.axvspan(noise_burst[0], noise_burst[1], alpha=0.15, label="noise burst")
plt.axvspan(model_mismatch[0], model_mismatch[1], alpha=0.10, label="model mismatch")
plt.title("Phase alignment recovery: threshold margin")
plt.xlabel("calibration block")
plt.ylabel("CGCS margin above threshold")
plt.legend(loc="lower left")
save_fig("threshold_margin")
plt.show()


In [ ]:
names = [n for n in ranking if n != "oracle"]
values = [metrics[n]["phase_alignment_error_integral"] for n in names]

plt.figure(figsize=(12, 6))
plt.bar(names, values)
plt.title("Phase alignment recovery: phase-error integral")
plt.ylabel("mean post-recovery error integral")
plt.xticks(rotation=25, ha="right")
save_fig("phase_error_integral")
plt.show()


In [ ]:
names = [n for n in ranking if n != "oracle"]
values = [metrics[n]["reentry_overshoot"] for n in names]

plt.figure(figsize=(12, 6))
plt.bar(names, values)
plt.title("Phase alignment recovery: re-entry overshoot")
plt.ylabel("mean overshoot/drop after re-entry")
plt.xticks(rotation=25, ha="right")
save_fig("reentry_overshoot")
plt.show()


In [ ]:
names = [n for n in ranking if n != "oracle"]
values = [metrics[n]["ringing_score"] for n in names]

plt.figure(figsize=(12, 6))
plt.bar(names, values)
plt.title("Phase alignment recovery: ringing score")
plt.ylabel("mean |second difference| after re-entry")
plt.xticks(rotation=25, ha="right")
save_fig("ringing_score")
plt.show()


In [ ]:
names = [n for n in ranking if n != "oracle"]
values = [metrics[n]["settling_blocks"] for n in names]

plt.figure(figsize=(12, 6))
plt.bar(names, values)
plt.title("Phase alignment recovery: settling blocks")
plt.ylabel("mean blocks to settle above 0.98")
plt.xticks(rotation=25, ha="right")
save_fig("settling_blocks")
plt.show()


In [ ]:
names = ranking
values = [metrics[n]["clean_recovery_score"] for n in names]

plt.figure(figsize=(12, 6))
plt.bar(names, values)
plt.title("Phase alignment recovery: clean recovery ranking")
plt.ylabel("clean recovery score")
plt.xticks(rotation=25, ha="right")
save_fig("clean_recovery_ranking")
plt.show()


In [ ]:
# Pick worst re-entry event among non-oracle policies by largest post-recovery error integral.
worst = None
for name in ["none", "moving_average", "scalar_kalman", "joint_kalman", "robust_gated_kalman", "constrained_mpc"]:
    for e in metrics[name]["events"]:
        if e["recovered_at"] is None:
            continue
        r = e["recovered_at"]
        end = min(N, r + 8)
        val = float(np.sum(np.maximum(0.0, 1.0 - cosines[name][r:end])))
        if worst is None or val > worst["value"]:
            worst = {"policy": name, "event": e, "value": val}

worst


In [ ]:
if worst is None:
    lo, hi = 20, 50
else:
    r = worst["event"]["recovered_at"]
    lo, hi = max(0, r - 8), min(N, r + 12)

plt.figure(figsize=(14, 6))
for name in ["none", "moving_average", "scalar_kalman", "joint_kalman", "robust_gated_kalman", "constrained_mpc"]:
    plt.plot(blocks[lo:hi], cosines[name][lo:hi], marker="o", label=name)
plt.axhline(threshold, linestyle="--", label="45° threshold")
plt.axvspan(noise_burst[0], noise_burst[1], alpha=0.15, label="noise burst")
plt.axvspan(model_mismatch[0], model_mismatch[1], alpha=0.10, label="model mismatch")
title_suffix = "" if worst is None else f" — worst policy: {worst['policy']}"
plt.title(f"Phase alignment recovery: worst re-entry window{title_suffix}")
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.ylim(0.55, 1.01)
plt.legend(loc="lower left")
save_fig("worst_reentry_window")
plt.show()


In [ ]:
lo = max(0, noise_burst[0] - 4)
hi = min(N, model_mismatch[1] + 6)

plt.figure(figsize=(14, 6))
for name in ["scalar_kalman", "joint_kalman", "robust_gated_kalman", "constrained_mpc"]:
    plt.plot(blocks[lo:hi], rmse[name][lo:hi], marker="o", label=name)
plt.axvspan(noise_burst[0], noise_burst[1], alpha=0.15, label="noise burst")
plt.axvspan(model_mismatch[0], model_mismatch[1], alpha=0.10, label="model mismatch")
plt.title("Phase alignment recovery: disturbance-window zoom")
plt.xlabel("calibration block")
plt.ylabel("RMSE vs target response")
plt.legend(loc="upper right")
save_fig("disturbance_window_zoom")
plt.show()


## 5. Results exports


In [ ]:
# Build compact results table.
rows = []
for name in ranking:
    m = metrics[name]
    rows.append({
        "policy": name,
        "failure_count": m["failure_count"],
        "recovery_count": m["recovery_count"],
        "time_below_threshold": m["time_below_threshold"],
        "mean_threshold_margin": m["mean_threshold_margin"],
        "post_recovery_cosine_slope": m["post_recovery_cosine_slope"],
        "phase_alignment_error_integral": m["phase_alignment_error_integral"],
        "reentry_overshoot": m["reentry_overshoot"],
        "ringing_score": m["ringing_score"],
        "settling_blocks": m["settling_blocks"],
        "clean_recovery_score": m["clean_recovery_score"],
        "mean_response_rmse": float(np.mean(rmse[name])),
        "max_response_rmse": float(np.max(rmse[name])),
        "min_cosine": float(np.min(cosines[name])),
    })

summary_df = pd.DataFrame(rows)
summary_df


In [ ]:
summary = {
    "notebook": "14_phase_alignment_recovery",
    "description": "Measures recovery quality after CGCS phase-lock re-entry.",
    "configuration": {
        "blocks": N,
        "threshold": threshold,
        "noise_burst": list(noise_burst),
        "model_mismatch": list(model_mismatch),
        "shock_blocks": shock_blocks,
        "policies": list(policies.keys()),
    },
    "ranking": ranking,
    "metrics": {name: {k: v for k, v in metrics[name].items() if k != "events"} for name in metrics},
    "events": {name: metrics[name]["events"] for name in metrics},
    "figures": [
        f"{PREFIX}_cgcs_stability_comparison.png",
        f"{PREFIX}_threshold_margin.png",
        f"{PREFIX}_phase_error_integral.png",
        f"{PREFIX}_reentry_overshoot.png",
        f"{PREFIX}_ringing_score.png",
        f"{PREFIX}_settling_blocks.png",
        f"{PREFIX}_clean_recovery_ranking.png",
        f"{PREFIX}_worst_reentry_window.png",
        f"{PREFIX}_disturbance_window_zoom.png",
    ],
}

json_path = f"{RESULTS_DIR}/{SLUG}_summary.json"
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

csv_path = f"{RESULTS_DIR}/{SLUG}_summary.csv"
summary_df.to_csv(csv_path, index=False)

print(f"[saved] {json_path}")
print(f"[saved] {csv_path}")


In [ ]:
def md_float(x, digits=4):
    return f"{x:.{digits}f}"

policy_lines = []
for i, row in enumerate(rows, start=1):
    policy_lines.append(
        f"| {i} | {row['policy']} | {row['clean_recovery_score']:.4f} | "
        f"{row['failure_count']} | {row['time_below_threshold']} | "
        f"{row['phase_alignment_error_integral']:.4f} | {row['ringing_score']:.4f} | "
        f"{row['settling_blocks']:.2f} |"
    )

results_md = f"""# Phase Alignment Recovery Results Summary

## Configuration

- Blocks: {N}
- Noise burst window: {noise_burst[0]}–{noise_burst[1]}
- Model mismatch window: {model_mismatch[0]}–{model_mismatch[1]}
- Shock blocks: {", ".join(map(str, shock_blocks))}
- Phase-lock threshold: {threshold:.4f}

---

## Policy Ranking

| Rank | Policy | Clean Recovery Score | Failures | Blocks Below Threshold | Phase Error Integral | Ringing Score | Settling Blocks |
|------|--------|----------------------|----------|------------------------|----------------------|---------------|-----------------|
{chr(10).join(policy_lines)}

---

## Key Observations

- Notebook 13 measured recovery time.
- Notebook 14 measures recovery quality after re-entry.
- Clean recovery rewards high threshold margin and penalizes:
  - repeated failures
  - longer time below threshold
  - post-recovery phase error
  - re-entry overshoot
  - ringing
  - delayed settling

---

## Interpretation

Clean recovery is not identical to fast recovery.

A controller can return above threshold quickly and still re-enter with overshoot or ringing.
This notebook separates recovery duration from recovery quality.

---

## Next Direction

Notebook 15 should consolidate controller selection into a deployment scorecard:

- accuracy
- robustness
- recovery time
- recovery quality
- command smoothness
- phase-lock margin
"""

results_md_path = f"{RESULTS_DIR}/{SLUG}_summary.md"
with open(results_md_path, "w") as f:
    f.write(results_md)

print(f"[saved] {results_md_path}")


In [ ]:
figure_blocks = [
    ("CGCS phase-lock stability", "cgcs_stability_comparison", "Shows whether each policy remains above the 45° CGCS threshold."),
    ("Threshold margin", "threshold_margin", "Shows signed distance above or below the phase-lock threshold."),
    ("Phase-error integral", "phase_error_integral", "Measures accumulated post-recovery phase error after re-entry."),
    ("Re-entry overshoot", "reentry_overshoot", "Measures re-entry instability after returning above threshold."),
    ("Ringing score", "ringing_score", "Measures post-recovery oscillation using second differences."),
    ("Settling blocks", "settling_blocks", "Measures blocks needed to settle above high-alignment behavior."),
    ("Clean recovery ranking", "clean_recovery_ranking", "Ranks policies by integrated recovery-quality score."),
    ("Worst re-entry window", "worst_reentry_window", "Zooms around worst post-recovery behavior."),
    ("Disturbance-window zoom", "disturbance_window_zoom", "Compares recovery quality in noise and mismatch regions."),
]

figure_sections = "\n\n".join(
    [
        f"### {title}\n\n![{title}](../figures/{SLUG}/{PREFIX}_{name}.png)\n\n{caption}"
        for title, name, caption in figure_blocks
    ]
)

docs_md = f"""# Phase Alignment Recovery

Notebook 14 extends failure-boundary and recovery-time analysis into **recovery quality**.

The core question:

> After returning above the 45° CGCS threshold, does the policy recover cleanly?

---

## Model

CGCS phase-lock condition:

```text
cosθ ≥ 1 / √(1² + 1²)
```

Notebook 14 evaluates post-recovery behavior after a policy re-enters this valid region.

---

## Metrics

- `post_recovery_cosine_slope`
- `phase_alignment_error_integral`
- `reentry_overshoot`
- `ringing_score`
- `settling_blocks`
- `clean_recovery_score`

Clean recovery penalizes repeated failures, long time below threshold, high phase error, overshoot, ringing, and delayed settling.

---

## Figures

{figure_sections}

---

## Key Takeaways

- Recovery quality is distinct from recovery speed.
- Kalman-family policies generally reduce phase error after re-entry.
- Robust gated Kalman can suppress outlier-driven re-entry artifacts.
- Constrained MPC can smooth command behavior, though excessive smoothing may delay alignment.
- Moving average and no-control policies show more lag and weaker re-entry stability.

---

## Next Step

Notebook 15:

- Build a deployment scorecard.
- Combine accuracy, robustness, phase-lock margin, recovery time, clean recovery, and command smoothness.
"""

docs_md_path = f"{DOCS_DIR}/{SLUG}.md"
with open(docs_md_path, "w") as f:
    f.write(docs_md)

print(f"[saved] {docs_md_path}")


## 6. Optional Colab download zip

Run this final cell in Colab when you want a single downloadable artifact.


In [ ]:
# Optional Colab export:
# !zip -r notebook14_phase_alignment_recovery_outputs.zip figures/phase_alignment_recovery results docs
# from google.colab import files
# files.download("notebook14_phase_alignment_recovery_outputs.zip")
